In [ ]:
# =====================================================================================
# セル1: ライブラリの準備と設定
# =====================================================================================
import pandas as pd
import matplotlib.pyplot as plt
import os
import io
import re
import tkinter as tk
from tkinter import filedialog

# --- 設定：日本語フォントおよびグラフデザイン ---
def set_japanese_font():
    standard_fonts = ['Yu Gothic', 'Meiryo', 'MS Gothic', 'Hiragino Sans', 'IPAexGothic']
    for font in standard_fonts:
        try:
            plt.rcParams['font.family'] = font
            plt.rcParams['axes.unicode_minus'] = False
            return
        except:
            continue
set_japanese_font()

# マーカーと色の設定
TEMP_MARKERS = {190: 'o', 210: 's', 230: '^', 250: 'D', 280: 'v'}
THICK_MARKERS = {0.5: 'o', 1.0: 's', 1.5: '^', 2.0: 'D', 2.5: 'v', 3.0: 'p'}
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# グローバル変数の初期化
file_paths = []
dataframes = {}

print("セル1の実行が完了しました。ライブラリと設定が読み込まれました。")

In [ ]:
# =====================================================================================
# セル2: ファイル選択とデータの集計処理
# =====================================================================================
print("--- ステップ1: ファイル選択 ---")
root = tk.Tk()
root.withdraw()
root.call('wm', 'attributes', '.', '-topmost', True) # ダイアログを最前面に

file_paths = filedialog.askopenfilenames(
    title="解析したいCSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

if not file_paths:
    print("ファイルが選択されませんでした。処理を中断します。")
else:
    # 必要なカラムの定義
    x_col = '射出速度[mm/s]'
    thick_col = 'キャビティ厚み[mm]'
    temp_col = '加熱シリンダ温度[℃]'
    
    y_cols = [
        '(1)HA-V05_最高値(℃)',
        '(1)HA-V05_最高値の時間(s)',
        '(1)HA-V05_0.7秒前の値(V)',
        '(1)HA-V05_0.7秒前の時間(s)',
        '(1)HA-V05_立ち上がり値(V)',
        '(1)HA-V05_立ち上がり時間(s)'
    ]
    
    # ファイルによって名前が異なるカラムの対応表（別名 → 上の標準名）
    # mixppは「最高値(V)」が電圧のため、換算後の℃を最高値として使う
    COLUMN_ALIASES = {
        '最高値換算結果(℃)': '(1)HA-V05_最高値(℃)',
        '(1)HA-V05_抽出ピーク時間(s)': '(1)HA-V05_最高値の時間(s)',
    }
    
    # 立ち上がりからピークまでの時間がこれを超える行はピーク抽出失敗の疑いとして表示する
    PEAK_GAP_LIMIT = 0.5  # [s]

    for file_path in file_paths:
        print(f"\n■ 読み込み開始: {os.path.basename(file_path)}")
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='shift_jis')
        
        # 別名のカラムを標準名にそろえる
        df = df.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in df.columns and v not in df.columns})
            
        # 必須カラムの存在チェック
        missing_cols = [c for c in [x_col, thick_col, temp_col] + y_cols if c not in df.columns]
        if missing_cols:
            print(f"警告: 以下のカラムが見つかりません。スキップします。\n{missing_cols}")
            continue
        
        # 異常値の確認（除外はしない）
        peak_gap = df['(1)HA-V05_最高値の時間(s)'] - df['(1)HA-V05_立ち上がり時間(s)']
        suspicious = df[peak_gap > PEAK_GAP_LIMIT]
        if not suspicious.empty:
            print(f"注意: 立ち上がりからピークまでが{PEAK_GAP_LIMIT}秒を超える行が{len(suspicious)}行あります（ピーク抽出失敗の疑い）")
            print(suspicious[['元ファイル', thick_col, temp_col, x_col]].to_string(index=False))
            
        print("--- ステップ2: データ集計 ---")
        agg_funcs = {col: ['mean', 'std'] for col in y_cols}
        df_agg = df.groupby([thick_col, temp_col, x_col]).agg(agg_funcs).reset_index()
        
        # カラム名のフラット化
        new_columns = []
        for col in df_agg.columns:
            if col[1] == '':
                new_columns.append(col[0])
            else:
                new_columns.append(f"{col[0]}_{col[1]}")
        df_agg.columns = new_columns
        
        # 辞書に格納
        dataframes[file_path] = df_agg
        print(f"集計完了: {os.path.basename(file_path)}")
        
    if dataframes:
        print("\n全てのファイルの読み込みと集計が完了しました。次のセルへ進んでください。")

In [ ]:
# =====================================================================================
# セル3: Excelへのグラフ出力と保存
# =====================================================================================
if 'dataframes' not in locals() or not dataframes:
    print("エラー: 処理対象のデータがありません。セル2を先に実行してください。")
else:
    print("--- ステップ3: Excelおよびグラフの生成開始 ---")
    
    for file_path, df_agg in dataframes.items():
        output_dir = os.path.dirname(file_path)
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        excel_path = os.path.join(output_dir, f"グラフ出力結果_{base_name}.xlsx")
        
        unique_thicks = sorted(df_agg[thick_col].unique())
        unique_temps = sorted(df_agg[temp_col].unique())
        
        print(f"\n保存先: {excel_path}")
        
        with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
            workbook = writer.book
            header_format = workbook.add_format({'bold': True, 'align': 'left', 'bg_color': '#DDEBF7'})
            title_format = workbook.add_format({'bold': True, 'font_size': 14})
            
            for y_col in y_cols:
                print(f"  シート作成中: {y_col}")
                y_mean_col = f"{y_col}_mean"
                y_std_col = f"{y_col}_std"
                
                # Excelのシート名制約への対応
                sheet_name = re.sub(r'[\\/*?:\[\]]', '_', y_col)[:31]
                worksheet = workbook.add_worksheet(sheet_name)
                
                current_row = 1
                
                # --- セクション1: キャビティ厚みごと（単一グラフ ＆ 温度統合グラフ） ---
                worksheet.write(current_row, 0, "■ キャビティ厚みごとのグラフ", title_format)
                current_row += 2
                
                for thick in unique_thicks:
                    df_thick = df_agg[df_agg[thick_col] == thick]
                    
                    worksheet.merge_range(current_row, 0, current_row, 30, f"--- キャビティ厚み: {thick}mm ---", header_format)
                    current_row += 2
                    current_col = 1
                    
                    # 全温度統合グラフ（厚み別）の初期化
                    fig_sum, ax_sum = plt.subplots(figsize=(8, 5))
                    ax_sum.set_title(f"[厚み別 統合グラフ] {y_col}\n(キャビティ厚み: {thick}mm)")
                    ax_sum.set_xlabel(x_col)
                    ax_sum.set_ylabel(y_col)
                    ax_sum.grid(True)
                    
                    # 各温度ごとの単一グラフを作成
                    for i, temp in enumerate(unique_temps):
                        df_temp = df_thick[df_thick[temp_col] == temp].sort_values(x_col)
                        if df_temp.empty:
                            continue
                            
                        x = df_temp[x_col]
                        y_mean = df_temp[y_mean_col]
                        y_std = df_temp[y_std_col]
                        
                        color = COLORS[i % len(COLORS)]
                        marker = TEMP_MARKERS.get(temp, 'o')
                        
                        # 単一グラフ（エラーバー付き）
                        fig_ind, ax_ind = plt.subplots(figsize=(6, 4))
                        ax_ind.errorbar(x, y_mean, yerr=y_std, marker=marker, linestyle='-', color=color, capsize=4, label=f"{temp}℃")
                        ax_ind.set_title(f"[単一グラフ] {y_col}\n(キャビティ厚み: {thick}mm / 温度: {temp}℃)")
                        ax_ind.set_xlabel(x_col)
                        ax_ind.set_ylabel(y_col)
                        ax_ind.grid(True)
                        ax_ind.legend(frameon=False)
                        
                        fig_ind.tight_layout()
                        img_data_ind = io.BytesIO()
                        fig_ind.savefig(img_data_ind, format='png', dpi=120)
                        plt.close(fig_ind)
                        
                        # Excelに貼り付け
                        worksheet.write(current_row - 1, current_col, f"厚み{thick}mm_{temp}℃ (単一)")
                        worksheet.insert_image(current_row, current_col, '', {'image_data': img_data_ind})
                        
                        current_col += 10
                        
                        # 統合グラフ用に追加プロット
                        ax_sum.plot(x, y_mean, marker=marker, linestyle='-', color=color, label=f"{temp}℃")
                        
                    # 厚み別の統合グラフ出力
                    ax_sum.legend(title="加熱シリンダ温度", frameon=False)
                    fig_sum.tight_layout()
                    img_data_sum = io.BytesIO()
                    fig_sum.savefig(img_data_sum, format='png', dpi=120)
                    plt.close(fig_sum)
                    
                    worksheet.write(current_row - 1, current_col, f"厚み{thick}mm (全温度統合)")
                    worksheet.insert_image(current_row, current_col, '', {'image_data': img_data_sum})
                    
                    current_row += 23  
                    
                # --- セクション2: 加熱シリンダ温度ごと（全厚み統合グラフ） ---
                current_row += 2
                worksheet.write(current_row, 0, "■ 加熱シリンダ温度ごとのグラフ（全厚み統合）", title_format)
                current_row += 2
                
                current_col = 1
                for i, temp in enumerate(unique_temps):
                    df_temp = df_agg[df_agg[temp_col] == temp]
                    
                    # 全厚み統合グラフ（温度別）の初期化
                    fig_sum_t, ax_sum_t = plt.subplots(figsize=(8, 5))
                    ax_sum_t.set_title(f"[温度別 統合グラフ] {y_col}\n(加熱シリンダ温度: {temp}℃)")
                    ax_sum_t.set_xlabel(x_col)
                    ax_sum_t.set_ylabel(y_col)
                    ax_sum_t.grid(True)
                    
                    # 厚みごとのプロットを追加
                    for j, thick in enumerate(unique_thicks):
                        df_plot = df_temp[df_temp[thick_col] == thick].sort_values(x_col)
                        if df_plot.empty:
                            continue
                            
                        x = df_plot[x_col]
                        y_mean = df_plot[y_mean_col]
                        
                        color = COLORS[j % len(COLORS)]
                        marker = THICK_MARKERS.get(thick, 's')
                        
                        ax_sum_t.plot(x, y_mean, marker=marker, linestyle='-', color=color, label=f"{thick}mm")
                    
                    ax_sum_t.legend(title="キャビティ厚み", frameon=False)
                    fig_sum_t.tight_layout()
                    img_data_sum_t = io.BytesIO()
                    fig_sum_t.savefig(img_data_sum_t, format='png', dpi=120)
                    plt.close(fig_sum_t)
                    
                    # Excelに貼り付け
                    worksheet.write(current_row - 1, current_col, f"温度{temp}℃ (全厚み統合)")
                    worksheet.insert_image(current_row, current_col, '', {'image_data': img_data_sum_t})
                    
                    current_col += 13
                    
        print(f"'{base_name}' の処理が完全に完了しました。")